In [5]:
import duckdb


In [6]:
con = duckdb.connect(database='dados_duckdb.db ', read_only=False)

In [7]:
df = con.execute("""
                  select * 
                  from (
                  select * , row_number() over (partition by NATBR order by data_ingestao desc) as row
                  from bronze_z0019
                  where data_ingestao >= '2026-11-18'
                  ) where  row = 1  
                 """) .fetchdf()
df.head(10)


,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-11-18 19:57:45.688483,1
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-11-18 19:57:45.688483,1
2,10004,SERRA,BT50,100,200,z0019_2.csv,2026-11-18 19:58:47.461714,1
3,10005,CHAVE,BT10,100,300,z0019_2.csv,2026-11-18 19:58:47.461714,1
4,10003,PREGO,BT10,100,60,z0019_2.csv,2026-11-18 19:58:47.461714,1


In [11]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row'])
df_final = df_final.rename(columns={"NATBR" : "id"})
df_final = df_final.rename(columns={"MAKTX" : "nm_produto"})
df_final = df_final.rename(columns={"WERKS" : "id_categoria"})
df_final = df_final.rename(columns={"MAINS" : "id_fornecedor"})
df_final = df_final.rename(columns={"LABST" : "vl_preco"})
df_final.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10004,SERRA,BT50,100,200
3,10005,CHAVE,BT10,100,300
4,10003,PREGO,BT10,100,60


In [12]:
df_final.dtypes

id                 str
nm_produto         str
id_categoria       str
id_fornecedor    int32
vl_preco         int32
dtype: object

In [ ]:
df2 = df_final
df2 = df2.astype(
    {
        'id' : int,
        'nm_produto' : str,
        'id_categoria' : str,
        'id_fornecedor' : int,
        'vl_preco' : float
    }
)
df2.dtypes


id                 int64
nm_produto           str
id_categoria         str
id_fornecedor      int64
vl_preco         float64
dtype: object

In [17]:
con.execute("""
    CREATE TABLE IF NOT EXISTS produtos (
    id BIGINT,
    nm_produto VARCHAR,
    id_categoria VARCHAR,
    id_fornecedor BIGINT,
    vl_preco FLOAT
    )
    """)

In [18]:
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,CHAVE,BT10,100,300.0
4,10003,PREGO,BT10,100,60.0


In [20]:
con.execute("insert into produtos select * from df2")

In [21]:
df_resultado = con.execute("select * from produtos").fetchdf()
df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,CHAVE,BT10,100,300.0
4,10003,PREGO,BT10,100,60.0


In [22]:
con.close()